In [1]:
# Install necessary libraries for Transformers (PyTorch) and Keras
!pip install -q -U torch transformers
!pip install -q -U keras>=3 keras-nlp
!pip install -q -U datasets colorama nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 161.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [2]:
import os
import random
import torch
import nltk
import keras
import keras_nlp
import numpy as np
import tensorflow as tf
from transformers import GPT2Tokenizer, GPT2LMHeadModel, RobertaTokenizer, RobertaForMaskedLM
from nltk.corpus import stopwords
from colorama import Fore, Style, init

# Set backend to avoid conflicts, prioritizing TensorFlow for the Keras model
os.environ["KERAS_BACKEND"] = "tensorflow"

# Initialize utilities
init(autoreset=True)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Setup device for PyTorch (RoBERTa models)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch using device: {device}")

PyTorch using device: cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
class Logging:
    """Simple logging class to track generation steps."""
    def __init__(self, log_file="generation.log"):
        self.log_file = log_file
        # Clear file on init
        with open(self.log_file, "w") as f:
            pass

    def info(self, message):
        print(f"INFO: {message}")
        with open(self.log_file, "a") as f:
            f.write(f"INFO: {message}\n")

    def error(self, message):
        print(f"ERROR: {message}")
        with open(self.log_file, "a") as f:
            f.write(f"ERROR: {message}\n")

In [4]:
class HybridGenerator:
    def __init__(self, gpt_model, gpt_tokenizer, roberta_model, roberta_tokenizer, is_gpt_keras=False):
        self.gpt_model = gpt_model
        self.gpt_tokenizer = gpt_tokenizer
        self.roberta_model = roberta_model
        self.roberta_tokenizer = roberta_tokenizer
        self.is_gpt_keras = is_gpt_keras
        self.logger = Logging("hybrid_gen.log")

    def _gpt_generate_step(self, context_text, k_tokens, temperature=0.7):
        """Generates k new tokens using either Keras or PyTorch GPT-2."""
        if self.is_gpt_keras:
            # --- Keras Implementation (for Fine-Tuned GPT-2) ---
            # Encode
            input_ids = self.gpt_tokenizer.encode(context_text, return_tensors="tf")

            for _ in range(k_tokens):
                # Get logits
                outputs = self.gpt_model(input_ids)
                # outputs is usually [batch, seq, vocab]
                next_token_logits = outputs[:, -1, :] / temperature
                # Sample
                next_token_id = tf.random.categorical(next_token_logits, num_samples=1)
                # Append
                input_ids = tf.concat([input_ids, next_token_id], axis=-1)

            generated_text = self.gpt_tokenizer.decode(input_ids[0], skip_special_tokens=True)

        else:
            # --- PyTorch Implementation (for Base GPT-2) ---
            inputs = self.gpt_tokenizer(context_text, return_tensors='pt').to(device)
            self.gpt_model.to(device)

            outputs = self.gpt_model.generate(
                **inputs,
                max_new_tokens=k_tokens,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=temperature,
                pad_token_id=self.gpt_tokenizer.eos_token_id
            )
            generated_text = self.gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)

        return generated_text

    def _roberta_refine_step(self, text, start_index_words, mask_prob):
        """Masks tokens in the newly generated segment and refills them using RoBERTa."""
        if self.roberta_model is None:
            return text

        words = text.split()
        # Only consider words generated in this step (after start_index)
        new_segment_indices = range(start_index_words, len(words))

        if not new_segment_indices:
            return text

        # Select indices to mask based on probability and stopword constraint
        mask_indices = []
        for i in new_segment_indices:
            word = words[i]
            # Probability check + Stopword check
            if word.lower() not in stop_words and random.random() < mask_prob:
                mask_indices.append(i)

        if not mask_indices:
            return text

        # Apply masks to a copy of the words list
        masked_words = words.copy()
        for idx in mask_indices:
            masked_words[idx] = self.roberta_tokenizer.mask_token

        masked_text = " ".join(masked_words)
        # self.logger.info(f"Masked: ...{' '.join(masked_words[max(0, start_index_words):])}")

        # --- RoBERTa Prediction (PyTorch) ---
        inputs = self.roberta_tokenizer(masked_text, return_tensors="pt").to(device)
        self.roberta_model.to(device)

        with torch.no_grad():
            outputs = self.roberta_model(**inputs)
            logits = outputs.logits

        # Replace masked tokens with top prediction
        # Note: We need to align word indices to token indices.
        # For simplicity in this demo, we assume the mask token in 'inputs' corresponds to the word mask.

        # Find positions of mask tokens in the encoded tensor
        mask_token_id = self.roberta_tokenizer.mask_token_id
        input_ids = inputs.input_ids[0]
        mask_positions = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]

        # Iterate through masks and replace
        # We assume order of masks in 'words' matches order of masks in 'tokens'
        if len(mask_positions) == len(mask_indices):
            for i, word_idx in enumerate(mask_indices):
                token_pos = mask_positions[i]
                predicted_token_id = logits[0, token_pos].argmax(axis=-1)
                filled_word = self.roberta_tokenizer.decode(predicted_token_id, skip_special_tokens=True)
                masked_words[word_idx] = filled_word.strip()

        return " ".join(masked_words)

    def generate_song(self, prompt, max_length=100, k_step=5, mask_prob=0.3):
        current_text = prompt
        self.logger.info(f"Starting Generation. Prompt: {prompt}")

        while len(current_text.split()) < max_length:
            start_word_count = len(current_text.split())

            # 1. GPT Step (Generate k tokens)
            current_text = self._gpt_generate_step(current_text, k_step)

            # 2. RoBERTa Step (Refinement)
            # Only refine if we actually added words
            if len(current_text.split()) > start_word_count:
                current_text = self._roberta_refine_step(current_text, start_word_count, mask_prob)

        return current_text

In [5]:
# Load Standard GPT-2 (PyTorch)
print("Loading Base GPT-2...")
gpt2_base_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_base_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Load Standard RoBERTa (PyTorch)
print("Loading Base RoBERTa...")
roberta_base_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_base_model = RobertaForMaskedLM.from_pretrained('roberta-base')

Loading Base GPT-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [26]:
class HybridGenerator:
    def __init__(self, gpt_model, gpt_tokenizer, roberta_model, roberta_tokenizer, is_gpt_keras=False):
        self.gpt_model = gpt_model
        self.gpt_tokenizer = gpt_tokenizer
        self.roberta_model = roberta_model
        self.roberta_tokenizer = roberta_tokenizer
        self.is_gpt_keras = is_gpt_keras
        self.logger = Logging("hybrid_gen.log")

    def _gpt_generate_step(self, context_text, k_tokens, temperature=0.7):
        """Generates k new tokens using either Keras or PyTorch GPT-2."""
        if self.is_gpt_keras:
            # --- Keras Implementation (Fixed for Dict Input) ---
            input_ids = self.gpt_tokenizer.encode(context_text, return_tensors="tf")

            for _ in range(k_tokens):
                # 1. Create Padding Mask
                padding_mask = tf.ones_like(input_ids)

                # 2. Prepare Dictionary Input (Crucial Fix)
                inputs = {
                    "token_ids": input_ids,
                    "padding_mask": padding_mask
                }

                # 3. Get Logits
                outputs = self.gpt_model(inputs)

                # outputs is [batch, seq, vocab]
                next_token_logits = outputs[:, -1, :] / temperature

                # 4. Sample
                next_token_id = tf.random.categorical(next_token_logits, num_samples=1, dtype=tf.int32)

                # 5. Append
                input_ids = tf.concat([input_ids, next_token_id], axis=-1)

            generated_text = self.gpt_tokenizer.decode(input_ids[0], skip_special_tokens=True)

        else:
            # --- PyTorch Implementation ---
            inputs = self.gpt_tokenizer(context_text, return_tensors='pt').to(device)
            self.gpt_model.to(device)

            outputs = self.gpt_model.generate(
                **inputs,
                max_new_tokens=k_tokens,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=temperature,
                pad_token_id=self.gpt_tokenizer.eos_token_id
            )
            generated_text = self.gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)

        return generated_text

    def _roberta_refine_step(self, text, start_index_words, mask_prob):
        """Masks tokens in the newly generated segment and refills them using RoBERTa."""
        if self.roberta_model is None:
            return text

        words = text.split()
        new_segment_indices = range(start_index_words, len(words))

        if not new_segment_indices:
            return text

        # Select mask indices
        mask_indices = []
        for i in new_segment_indices:
            word = words[i]
            if word.lower() not in stop_words and random.random() < mask_prob:
                mask_indices.append(i)

        if not mask_indices:
            return text

        # Masking
        masked_words = words.copy()
        for idx in mask_indices:
            masked_words[idx] = self.roberta_tokenizer.mask_token

        masked_text = " ".join(masked_words)

        # --- RoBERTa Prediction (PyTorch) ---
        inputs = self.roberta_tokenizer(masked_text, return_tensors="pt").to(device)
        self.roberta_model.to(device)

        with torch.no_grad():
            outputs = self.roberta_model(**inputs)
            logits = outputs.logits

        # Replace masked tokens
        mask_token_id = self.roberta_tokenizer.mask_token_id
        input_ids = inputs.input_ids[0]
        mask_positions = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]

        if len(mask_positions) == len(mask_indices):
            for i, word_idx in enumerate(mask_indices):
                token_pos = mask_positions[i]
                predicted_token_id = logits[0, token_pos].argmax(axis=-1)
                filled_word = self.roberta_tokenizer.decode(predicted_token_id, skip_special_tokens=True)
                masked_words[word_idx] = filled_word.strip()

        return " ".join(masked_words)

    def generate_song(self, prompt, max_length=100, k_step=5, mask_prob=0.3):
        current_text = prompt
        self.logger.info(f"Starting Generation. Prompt: {prompt}")

        while len(current_text.split()) < max_length:
            start_word_count = len(current_text.split())

            current_text = self._gpt_generate_step(current_text, k_step)

            if len(current_text.split()) > start_word_count:
                current_text = self._roberta_refine_step(current_text, start_word_count, mask_prob)

        return current_text

In [27]:
# Generation Parameters
PROMPT = "I am a"
MAX_LEN = 60      # Total words to generate
K_TOKENS = 6      # Words to generate per step before refining
MASK_PROB = 0.4   # Probability (0.0 to 1.0) of a new word being replaced by RoBERTa

In [28]:
print(f"{Fore.BLUE}Running Scenario 1: Base GPT-2 Only{Style.RESET_ALL}")

gen_1 = HybridGenerator(
    gpt_model=gpt2_base_model,
    gpt_tokenizer=gpt2_base_tokenizer,
    roberta_model=None, # No refinement
    roberta_tokenizer=None,
    is_gpt_keras=False
)

result_1 = gen_1.generate_song(PROMPT, MAX_LEN, K_TOKENS, MASK_PROB)
print(f"\n{Fore.GREEN}Result 1:{Style.RESET_ALL}\n{result_1}")

Running Scenario 1: Base GPT-2 Only
INFO: Starting Generation. Prompt: I am a

Result 1:
I am a little concerned about what is being done in the U.S. to try and stop this. We are a country that does not want to be in a position where it is the victim of a terrorist attack."

The video of the attack, which took place near the White House and the Capitol, can be viewed here:

The White House and


In [29]:
print(f"{Fore.BLUE}Running Scenario 2: Base GPT-2 + Base RoBERTa{Style.RESET_ALL}")

gen_2 = HybridGenerator(
    gpt_model=gpt2_base_model,
    gpt_tokenizer=gpt2_base_tokenizer,
    roberta_model=roberta_base_model,
    roberta_tokenizer=roberta_base_tokenizer,
    is_gpt_keras=False
)

result_2 = gen_2.generate_song(PROMPT, MAX_LEN, K_TOKENS, MASK_PROB)
print(f"\n{Fore.GREEN}Result 2:{Style.RESET_ALL}\n{result_2}")

Running Scenario 2: Base GPT-2 + Base RoBERTa
INFO: Starting Generation. Prompt: I am a

Result 2:
I am a fan of this game It's been a while since I've played it, but I think I'll stick with it. I'm not a huge fan of the idea of trying to be a part of it, but the things I like are the controls, the characters, the story, the story, the soundtrack, the character designs, and the art, and I'm


In [30]:
print(f"{Fore.BLUE}Running Scenario 3: Base GPT-2 + Fine-Tuned RoBERTa{Style.RESET_ALL}")

if roberta_ft_model:
    gen_3 = HybridGenerator(
        gpt_model=gpt2_base_model,
        gpt_tokenizer=gpt2_base_tokenizer,
        roberta_model=roberta_ft_model,
        roberta_tokenizer=roberta_ft_tokenizer,
        is_gpt_keras=False
    )

    result_3 = gen_3.generate_song(PROMPT, MAX_LEN, K_TOKENS, MASK_PROB)
    print(f"\n{Fore.GREEN}Result 3:{Style.RESET_ALL}\n{result_3}")
else:
    print("Skipping Scenario 3 (Fine-tuned RoBERTa not loaded)")
    result_3 = "N/A"

Running Scenario 3: Base GPT-2 + Fine-Tuned RoBERTa
INFO: Starting Generation. Prompt: I am a

Result 3:
I am a professional, I have been working for the past few years and I'm getting ready to move on from that. I feel like I've really enjoyed what I've done and I'm really excited about what is next. I think the last couple of years have been really good for me. "I'm really excited about the future and the


In [31]:
print(f"{Fore.BLUE}Running Scenario 4: Fine-Tuned GPT-2 + Base RoBERTa{Style.RESET_ALL}")

if gpt2_ft_model:
    gen_4 = HybridGenerator(
        gpt_model=gpt2_ft_model,
        gpt_tokenizer=gpt2_ft_tokenizer,
        roberta_model=roberta_base_model,
        roberta_tokenizer=roberta_base_tokenizer,
        is_gpt_keras=True # IMPORTANT: Switch to Keras logic for GPT
    )

    result_4 = gen_4.generate_song(PROMPT, MAX_LEN, K_TOKENS, MASK_PROB)
    print(f"\n{Fore.GREEN}Result 4:{Style.RESET_ALL}\n{result_4}")
else:
    print("Skipping Scenario 4 (Fine-tuned GPT-2 model not loaded)")
    result_4 = "N/A"

Running Scenario 4: Fine-Tuned GPT-2 + Base RoBERTa
INFO: Starting Generation. Prompt: I am a

Result 4:
I am a line in a song so many lines that I : but you are in here thinking i should be . but i lately i didn't�before and i am still wondering why or when or when or when or when or when or when or when or when or when or when or when or when or when or


In [32]:
print(f"{Fore.BLUE}Running Scenario 5: Fine-Tuned GPT-2 + Fine-Tuned RoBERTa{Style.RESET_ALL}")

if gpt2_ft_model and roberta_ft_model:
    gen_5 = HybridGenerator(
        gpt_model=gpt2_ft_model,
        gpt_tokenizer=gpt2_ft_tokenizer,
        roberta_model=roberta_ft_model,
        roberta_tokenizer=roberta_ft_tokenizer,
        is_gpt_keras=True # IMPORTANT: Switch to Keras logic for GPT
    )

    result_5 = gen_5.generate_song(PROMPT, MAX_LEN, K_TOKENS, MASK_PROB)
    print(f"\n{Fore.GREEN}Result 5:{Style.RESET_ALL}\n{result_5}")
else:
    print("Skipping Scenario 5 (Models not loaded)")
    result_5 = "N/A"

Running Scenario 5: Fine-Tuned GPT-2 + Fine-Tuned RoBERTa
INFO: Starting Generation. Prompt: I am a

Result 5:
I am a singer , , and a vocal ist at the music store . i wanna be your electric guitar . i'm a fan of music i'm a singer and everyone's talking about my bass and here's a reason for me to play the guitar i think you should hear the song barnyard was built in december of the old year and so


In [33]:
import pandas as pd
from IPython.display import display, HTML

data = {
    "Configuration": [
        "Base GPT-2 Only",
        "Base GPT-2 + Base RoBERTa",
        "Base GPT-2 + FT RoBERTa",
        "FT GPT-2 + Base RoBERTa",
        "FT GPT-2 + FT RoBERTa"
    ],
    "Generated Lyrics": [
        result_1,
        result_2,
        result_3,
        result_4,
        result_5
    ]
}

df = pd.DataFrame(data)

# Display properly with line breaks for readability
pd.set_option('display.max_colwidth', None)
display(HTML(df.to_html().replace("\\n","<br>")))

,Configuration,Generated Lyrics
0,Base GPT-2 Only,"I am a little concerned about what is being done in the U.S. to try and stop this. We are a country that does not want to be in a position where it is the victim of a terrorist attack.""The video of the attack, which took place near the White House and the Capitol, can be viewed here:The White House and"
1,Base GPT-2 + Base RoBERTa,"I am a fan of this game It's been a while since I've played it, but I think I'll stick with it. I'm not a huge fan of the idea of trying to be a part of it, but the things I like are the controls, the characters, the story, the story, the soundtrack, the character designs, and the art, and I'm"
2,Base GPT-2 + FT RoBERTa,"I am a professional, I have been working for the past few years and I'm getting ready to move on from that. I feel like I've really enjoyed what I've done and I'm really excited about what is next. I think the last couple of years have been really good for me. ""I'm really excited about the future and the"
3,FT GPT-2 + Base RoBERTa,I am a line in a song so many lines that I : but you are in here thinking i should be . but i lately i didn't�before and i am still wondering why or when or when or when or when or when or when or when or when or when or when or when or when or when or
4,FT GPT-2 + FT RoBERTa,"I am a singer , , and a vocal ist at the music store . i wanna be your electric guitar . i'm a fan of music i'm a singer and everyone's talking about my bass and here's a reason for me to play the guitar i think you should hear the song barnyard was built in december of the old year and so"
